# 🧹 LAB D6 – Limpieza de Datos
## Fundamentos de Gestión de Datos | TECSUP 2026-I
**Docente:** Mg. Pilar Rocío Sayán Mejía  
**Semana:** 6 | **Unidad:** II — Python para Datos + Limpieza + ML Básico


## 🎯 Capacidad a Lograr
Implementar un pipeline de limpieza de datos aplicando técnicas de detección e imputación de valores faltantes, identificación y tratamiento de outliers, eliminación de duplicados y normalización de variables numéricas, guardando los resultados en SQLite.


---
## ⚙️ Paso 1: Carga del dataset desde SQLite y diagnóstico inicial


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("✅ Librerías importadas | Pandas:", pd.__version__)


In [ ]:
# Conectar a SQLite y crear BD de ejemplo con problemas de calidad intencionales
conn = sqlite3.connect(":memory:")

conn.executescript("""
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    id_cliente INTEGER,
    id_producto INTEGER,
    cantidad INTEGER,
    monto_total REAL,
    fecha_venta TEXT,
    ciudad TEXT
);
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT,
    email TEXT,
    edad INTEGER,
    ciudad TEXT,
    ingreso_mensual REAL
);
""")

import random, string
random.seed(42)

ciudades = ['Lima','Lima','lima','LIMA','Arequipa','Cusco','Trujillo',None]
clientes_data = [
    (1,'Ana Torres','ana@mail.com',28,'Lima',3200.0),
    (2,'Luis Ríos','luis@mail.com',35,'arequipa',4500.0),
    (3,'María Paz',None,None,'Cusco',2800.0),
    (4,'Carlos Vega','carlos@mail.com',42,'LIMA',5600.0),
    (5,'Rosa Llanos','rosa@mail.com',29,'Lima',3100.0),
    (6,'Pedro Salas','pedro@mail.com',29,'Lima',3100.0),  # duplicado de Rosa (misma edad/ingreso)
    (7,'Juan Reyes','juan@mail.com',-5,'Trujillo',2200.0),  # edad negativa
    (8,'Sofía Mora','sofia@mail.com',31,None,None),
    (9,'Diego Quispe','diego@mail.com',200,'Lima',3400.0),  # edad outlier
    (10,'Ana Torres','ana@mail.com',28,'Lima',3200.0),  # duplicado exacto
]

ventas_data = [
    (1,1,1,2,6400.0,'2025-01-10','Lima'),
    (2,2,2,1,4500.0,'2025-01-11','Arequipa'),
    (3,3,1,3,None,'2025-01-12','Cusco'),
    (4,4,3,1,800.0,'2025-01-13','Lima'),
    (5,5,2,2,9000.0,'2025-01-14','Lima'),  # outlier alto
    (6,1,1,2,6400.0,'2025-01-10','Lima'),  # duplicado
    (7,7,1,None,0.0,'2025-01-15','Trujillo'),
    (8,8,2,1,4500.0,None,'Lima'),
]

conn.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?)", clientes_data)
conn.executemany("INSERT INTO ventas VALUES (?,?,?,?,?,?,?)", ventas_data)
conn.commit()

df_cli = pd.read_sql_query("SELECT * FROM clientes", conn)
df_ven = pd.read_sql_query("SELECT * FROM ventas", conn)

print("=== DIAGNÓSTICO INICIAL ===")
print(f"Clientes: {len(df_cli)} filas | Ventas: {len(df_ven)} filas")
print("\n--- Clientes: valores nulos ---")
print(df_cli.isnull().sum())
print("\n--- Ventas: valores nulos ---")
print(df_ven.isnull().sum())


## 🔍 Paso 2: Detección e imputación de valores faltantes


In [ ]:
# PASO 2: Imputación de valores faltantes
print("=== IMPUTACIÓN DE VALORES FALTANTES ===\n")

# ── Clientes ──
print("Antes:", df_cli.isnull().sum().to_dict())

# Edad: mediana (hay outlier negativo)
mediana_edad = df_cli['edad'].median()
df_cli['edad'] = df_cli['edad'].fillna(mediana_edad)
print(f"  • edad → mediana: {mediana_edad}")

# Ingreso_mensual: media
media_ingreso = df_cli['ingreso_mensual'].mean()
df_cli['ingreso_mensual'] = df_cli['ingreso_mensual'].fillna(media_ingreso)
print(f"  • ingreso_mensual → media: {media_ingreso:.1f}")

# Ciudad: moda
moda_ciudad = df_cli['ciudad'].mode()[0]
df_cli['ciudad'] = df_cli['ciudad'].fillna(moda_ciudad)
print(f"  • ciudad → moda: {moda_ciudad}")

# Email: marcador explícito
df_cli['email'] = df_cli['email'].fillna('sin_email@nd.com')

print("\nDespués:", df_cli.isnull().sum().to_dict())

# ── Ventas ──
df_ven['monto_total'] = df_ven['monto_total'].fillna(df_ven['monto_total'].median())
df_ven['cantidad'] = df_ven['cantidad'].fillna(df_ven['cantidad'].median())
df_ven['fecha_venta'] = df_ven['fecha_venta'].fillna('2025-01-01')
print("\nVentas nulos restantes:", df_ven.isnull().sum().to_dict())
print("\n✅ Imputación completada")


## 📊 Paso 3: Detección y tratamiento de outliers


In [ ]:
# PASO 3: Outliers con IQR y z-score
print("=== DETECCIÓN DE OUTLIERS ===\n")

for col in ['edad','ingreso_mensual']:
    Q1 = df_cli[col].quantile(0.25)
    Q3 = df_cli[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = df_cli[(df_cli[col] < lim_inf) | (df_cli[col] > lim_sup)]
    print(f"[{col}] Rango IQR: ({lim_inf:.1f}, {lim_sup:.1f}) | Outliers: {len(outliers)}")
    for _, row in outliers.iterrows():
        print(f"   → id={row['id_cliente']} | {col}={row[col]} | nombre={row['nombre']}")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ['edad','ingreso_mensual']):
    df_cli.boxplot(column=col, ax=ax)
    ax.set_title(f'Boxplot: {col}')
plt.tight_layout()
plt.savefig('/tmp/outliers_boxplot.png', dpi=100)
plt.show()

# Corregir outliers en edad (límite: 18-80)
df_cli.loc[df_cli['edad'] < 0, 'edad'] = mediana_edad
df_cli.loc[df_cli['edad'] > 80, 'edad'] = mediana_edad
print("\n✅ Outliers corregidos en 'edad'")


## 🔄 Paso 4: Eliminación de duplicados y corrección de tipos


In [ ]:
# PASO 4: Duplicados y tipos de datos
print("=== DUPLICADOS Y TIPOS ===\n")

print(f"Clientes duplicados exactos: {df_cli.duplicated().sum()}")
print(f"Emails duplicados: {df_cli.duplicated(subset=['email']).sum()}")

df_cli = df_cli.drop_duplicates(subset=['email'], keep='first')
print(f"Clientes después de deduplicación: {len(df_cli)}")

# Normalizar ciudad a título
df_cli['ciudad'] = df_cli['ciudad'].str.strip().str.title()
print("\nCiudades únicas después de normalizar:", df_cli['ciudad'].unique().tolist())

# Corrección de tipos
df_cli['edad'] = df_cli['edad'].astype(int)
df_cli['id_cliente'] = df_cli['id_cliente'].astype(int)

# Ventas
df_ven = df_ven.drop_duplicates()
df_ven['monto_total'] = df_ven['monto_total'].astype(float)
print(f"\nVentas duplicadas eliminadas → {len(df_ven)} registros")

print("\nTipos finales clientes:")
print(df_cli.dtypes.to_string())


## 📐 Paso 5: Normalización y estandarización


In [ ]:
# PASO 5: Min-Max y Z-score
print("=== NORMALIZACIÓN ===\n")

cols_num = ['edad','ingreso_mensual']

for col in cols_num:
    # Min-Max
    mn, mx = df_cli[col].min(), df_cli[col].max()
    df_cli[f'{col}_minmax'] = (df_cli[col] - mn) / (mx - mn)
    # Z-score
    mu, sigma = df_cli[col].mean(), df_cli[col].std()
    df_cli[f'{col}_zscore'] = (df_cli[col] - mu) / sigma
    print(f"[{col}]  Min-Max range: [{df_cli[col+'_minmax'].min():.2f}, {df_cli[col+'_minmax'].max():.2f}]"
          f"  |  Z-score mean: {df_cli[col+'_zscore'].mean():.2f}")

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for i, col in enumerate(cols_num):
    df_cli[col].hist(ax=axes[i][0], bins=6, color='steelblue', edgecolor='black')
    axes[i][0].set_title(f'{col} — Original')
    df_cli[f'{col}_minmax'].hist(ax=axes[i][1], bins=6, color='coral', edgecolor='black')
    axes[i][1].set_title(f'{col} — Min-Max')
plt.tight_layout()
plt.savefig('/tmp/normalizacion.png', dpi=100)
plt.show()
print("\n✅ Normalización completada")


## 💾 Paso 6: Guardado del dataset limpio en SQLite


In [ ]:
# PASO 6: Guardar datos limpios
print("=== GUARDADO EN SQLITE ===\n")

df_cli.to_sql('clientes_limpios', conn, if_exists='replace', index=False)
df_ven.to_sql('ventas_limpias', conn, if_exists='replace', index=False)
conn.commit()

# Reporte comparativo
reporte = {
    'Clientes originales': 10, 'Clientes limpios': len(df_cli),
    'Ventas originales': 8,    'Ventas limpias': len(df_ven),
    'Nulos imputados': 6,      'Outliers corregidos': 2,
    'Duplicados eliminados': 3
}
print("📋 REPORTE DE CALIDAD:")
for k,v in reporte.items():
    print(f"  {k:30s}: {v}")

# Verificar tablas guardadas
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("\n🗄️  Tablas en SQLite:", [r[0] for r in cursor.fetchall()])
print("\n🎉 ¡Pipeline de limpieza completado!")


---
## 💬 Actividad 3 – Análisis Reflexivo

**A)** ¿Cuál es la diferencia entre eliminar un outlier y transformarlo? ¿En qué situaciones empresariales sería peligroso eliminarlo sin analizarlo?

> _Tu respuesta aquí_

**B)** Compara la imputación por media, mediana y moda. ¿Cuándo usarías cada una?

> _Tu respuesta aquí_

**C)** ¿Por qué guardar el dataset limpio como tabla separada y no sobrescribir los datos originales?

> _Tu respuesta aquí_

**D)** Después del pipeline, ¿qué porcentaje de datos conservaste? ¿Hubo decisiones difíciles?

> _Tu respuesta aquí_


---
## ✅ Conclusiones

1. _Conclusión sobre la detección e imputación de valores faltantes_
2. _Conclusión sobre el tratamiento de outliers y normalización_
3. _Conclusión sobre la importancia de documentar el pipeline de limpieza_


---
## 📚 Material Complementario
| Recurso | Enlace |
|---------|--------|
| Pandas – fillna, dropna | https://pandas.pydata.org/docs/ |
| Scipy Stats | https://docs.scipy.org/doc/scipy/reference/stats.html |
| Scikit-learn Preprocessing | https://scikit-learn.org/stable/modules/preprocessing.html |
| Towards Data Science – Data Cleaning | https://towardsdatascience.com/data-cleaning |
